# 06 · Final pipeline

Builds on the comparisons in 02–05, where YOLOv8s at 1280 px was the best detector. Each stage changes **one** thing
and is scored on all val frames with the same metric, so the effect of data / inference changes is kept apart from
the effect of model changes:

| stage | what changes | training? |
|---|---|---|
| `inference` | inference strategy on the fixed `yolov8s_1280` weights: higher input size, TTA, tiles (SAHI-style) | no |
| `temporal` | post-processing that uses neighbouring frames (bonus: data temporality) | no |
| `ensemble` | Weighted Boxes Fusion of YOLO and Faster R-CNN predictions | no |
| `nomask` | data: train without painting ignored regions | yes |
| `yolov8m_1280` | model: YOLOv8m instead of YOLOv8s, same protocol | yes |
| `final` | best model, trained longer, with the best inference and post-processing | yes |
| `rewbf` | WBF of the 1280 YOLO predictions with Faster R-CNN (after the leaderboard check) | no |

Inputs to attach (Add Input → Your Work): the output of `02_yolo` that contains `runs/yolov8s_1280`, and the output of
`03_faster_rcnn` (for the ensemble).

In [ ]:
!pip install -q ultralytics ensemble-boxes
import os, urllib.request
REPO = "https://raw.githubusercontent.com/MakcDutch/AI-homework2/main/"
for f in ["common.py", "pipeline.py"]:
    if not os.path.exists(f):
        urllib.request.urlretrieve(REPO + f, f)

In [ ]:
import time, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from ultralytics import YOLO
import common as C
import pipeline as P

ROOT = C.find_data_root()
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("work")
os.makedirs(WORK, exist_ok=True)
LOG = f"{WORK}/pipeline_results.csv"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

## Configuration
`STAGES` selects what runs in this session (one Kaggle session = 12 h max). `FINAL` is filled in after the
comparison stages.

In [ ]:
TRAIN_STEP = 5
EPOCHS = 12
DEVICE = 0
BASE_EXP = "yolov8s_1280"

STAGES = ["inference", "temporal", "ensemble", "nomask"]

# set from stages A-C and the public leaderboard: on val, 1536 inference was +0.006 better than 1280, but on the public
# test the same weights scored 0.2105 at 1536 vs 0.2533 at 1280 (test has 680x382 and 960x540 clips that train/val
# lack), so the final model predicts at its training size; temporal rescoring gave no real gain; WBF with
# Faster R-CNN (weights 2:1) gave the largest improvement
FINAL = dict(model="yolov8m.pt", imgsz=1280, batch=4, epochs=30, train_step=5,
             strategy="plain_1280", temporal_alpha=None, wbf_frcnn_weights=(2, 1))

In [ ]:
val_paths = C.list_frames(ROOT, "val")["image"].tolist()
test_paths = C.list_frames(ROOT, "test")["image"].tolist()
sizes = {**P.image_sizes(ROOT, "val"), **P.image_sizes(ROOT, "test")}

STRATEGIES = {
    "plain_1280": lambda m, p, sz=1280: C.ultralytics_predict(m, p, imgsz=sz, device=DEVICE),
    "plain_1536": lambda m, p, sz=1280: C.ultralytics_predict(m, p, imgsz=int(sz * 1.2), device=DEVICE, batch=8),
    "tta_1280":   lambda m, p, sz=1280: C.ultralytics_predict(m, p, imgsz=sz, device=DEVICE, augment=True, batch=8),
    "tiles_1280": lambda m, p, sz=1280: P.tiled_predict(m, p, imgsz=sz, device=DEVICE),
}


def log(stage, exp, val_preds, ms=None, **extra):
    metrics, _ = C.coco_evaluate(val_preds, ROOT, "val")
    row = dict(stage=stage, exp=exp, ms_per_img=None if ms is None else round(ms, 1), **extra, **metrics)
    C.results_table(LOG, row)
    print(exp, metrics)
    return metrics["mAP"]


def submit(name, val_preds, test_preds):
    C.save_preds(val_preds, f"{WORK}/val_preds_{name}.pkl")
    C.save_preds(test_preds, f"{WORK}/test_preds_{name}.pkl")
    C.make_submission(test_preds, ROOT, f"{WORK}/sub_{name}.csv")


def train_yolo(name, model, imgsz, epochs, batch, train_step=TRAIN_STEP, mask_ignore=True):
    # same training call as 02_yolo; Ultralytics accumulates gradients to a nominal batch of 64,
    # so a smaller `batch` for a bigger model keeps the effective batch size
    data_dir = f"/tmp/yolo_data_step{train_step}" + ("" if mask_ignore else "_nomask")
    data_yaml = C.build_yolo_dataset(ROOT, data_dir, train_step=train_step, val_step=5, mask_ignore=mask_ignore)
    m = YOLO(model)
    t0 = time.time()
    m.train(data=data_yaml, imgsz=imgsz, epochs=epochs, batch=batch, device=DEVICE, seed=C.SEED, workers=4,
            project=f"{WORK}/runs", name=name, exist_ok=True, verbose=False)
    return YOLO(f"{m.trainer.save_dir}/weights/best.pt"), round((time.time() - t0) / 3600, 2)

## A. Inference strategies on fixed weights
The `yolov8s_1280` weights from `02_yolo` are reused, so only the way predictions are produced changes.

In [ ]:
if "inference" in STAGES:
    base = YOLO(P.find_input(f"**/runs/{BASE_EXP}/weights/best.pt"))
    val_by_strategy = {}
    for s, fn in STRATEGIES.items():
        preds, ms = fn(base, val_paths)
        val_by_strategy[s] = preds
        log("inference", f"{BASE_EXP}+{s}", preds, ms)
    scores = C.results_table(LOG).query("stage == 'inference'").set_index("exp")["mAP"]
    BEST_STRATEGY = scores.idxmax().split("+")[1]
    print("best strategy on val:", BEST_STRATEGY)
    best_val = val_by_strategy[BEST_STRATEGY]
    best_test, _ = STRATEGIES[BEST_STRATEGY](base, test_paths)
    submit(f"{BASE_EXP}_{BEST_STRATEGY}", best_val, best_test)

## B. Temporal post-processing (bonus: data temporality)
Detections that are confirmed by an overlapping same-class box in the previous / next frame get a higher score,
isolated ones a lower score. `alpha` is chosen on val.

In [ ]:
if "temporal" in STAGES:
    for alpha in [0.2, 0.4, 0.6]:
        log("temporal", f"{BASE_EXP}+{BEST_STRATEGY}+temporal{alpha}", P.temporal_rescore(best_val, alpha), alpha=alpha)
    t = C.results_table(LOG).query("stage == 'temporal'").set_index("alpha")["mAP"]
    BEST_ALPHA = float(t.idxmax())
    submit(f"{BASE_EXP}_{BEST_STRATEGY}_temporal", P.temporal_rescore(best_val, BEST_ALPHA),
           P.temporal_rescore(best_test, BEST_ALPHA))

## C. Ensemble with Faster R-CNN (Weighted Boxes Fusion)

In [ ]:
if "ensemble" in STAGES:
    fr_val = C.load_preds(P.find_input("**/val_preds_frcnn_base.pkl"))
    fr_test = C.load_preds(P.find_input("**/test_preds_frcnn_base.pkl"))
    for w in [(2, 1), (1, 1)]:
        ens = P.wbf_ensemble([best_val, fr_val], list(w), sizes)
        log("ensemble", f"WBF yolo:frcnn={w[0]}:{w[1]}", ens, weights=f"{w[0]}:{w[1]}")
    e = C.results_table(LOG).query("stage == 'ensemble'").set_index("weights")["mAP"]
    w = tuple(int(v) for v in e.idxmax().split(":"))
    submit("wbf_yolo_frcnn", P.wbf_ensemble([best_val, fr_val], list(w), sizes),
           P.wbf_ensemble([best_test, fr_test], list(w), sizes))

## D. Data strategy: no masking of ignored regions
Same model and protocol as `yolov8s_1280` in `02_yolo` (val mAP there: see the results table), only the training
frames differ: ignored regions are left as they are.

In [ ]:
if "nomask" in STAGES:
    model, hours = train_yolo("yolov8s_1280_nomask", "yolov8s.pt", 1280, EPOCHS, 8, mask_ignore=False)
    v, ms = STRATEGIES["plain_1280"](model, val_paths)
    log("data", "yolov8s_1280_nomask+plain_1280", v, ms, train_hours=hours)
    t, _ = STRATEGIES["plain_1280"](model, test_paths)
    submit("yolov8s_1280_nomask", v, t)

## E. Model change: YOLOv8m, same protocol

In [ ]:
if "yolov8m_1280" in STAGES:
    model, hours = train_yolo("yolov8m_1280", "yolov8m.pt", 1280, EPOCHS, 4)
    v, ms = STRATEGIES["plain_1280"](model, val_paths)
    log("model", "yolov8m_1280+plain_1280", v, ms, train_hours=hours)
    t, _ = STRATEGIES["plain_1280"](model, test_paths)
    submit("yolov8m_1280", v, t)

## F. Final model
Chosen from the stages above; trained longer than the 12-epoch comparison budget.

In [ ]:
if "final" in STAGES:
    model, hours = train_yolo("final", FINAL["model"], FINAL["imgsz"], FINAL["epochs"], FINAL["batch"],
                              train_step=FINAL["train_step"])
    fn = STRATEGIES[FINAL["strategy"]]
    v, ms = fn(model, val_paths, FINAL["imgsz"])
    log("final", f"final+{FINAL['strategy']}", v, ms, train_hours=hours)
    t, _ = fn(model, test_paths, FINAL["imgsz"])
    submit("final", v, t)
    if FINAL["temporal_alpha"]:
        a = FINAL["temporal_alpha"]
        v2, t2 = P.temporal_rescore(v, a), P.temporal_rescore(t, a)
        log("final", f"final+{FINAL['strategy']}+temporal{a}", v2, alpha=a, train_hours=hours)
        submit("final_temporal", v2, t2)
    if FINAL["wbf_frcnn_weights"]:
        w = list(FINAL["wbf_frcnn_weights"])
        fr_val = C.load_preds(P.find_input("**/val_preds_frcnn_base.pkl"))
        fr_test = C.load_preds(P.find_input("**/test_preds_frcnn_base.pkl"))
        v3, t3 = P.wbf_ensemble([v, fr_val], w, sizes), P.wbf_ensemble([t, fr_test], w, sizes)
        log("final", f"final+{FINAL['strategy']}+WBF frcnn {w[0]}:{w[1]}", v3, weights=f"{w[0]}:{w[1]}", train_hours=hours)
        submit("final_wbf", v3, t3)

## G. Re-ensembling at 1280 (CPU only)
The public score showed that 1536 inference hurts on test (0.2533 at 1280 → 0.2105 at 1536 for the same weights),
while WBF with Faster R-CNN helped (+0.03 over the same YOLO predictions). This stage fuses the existing 1280 predictions
of `yolov8s_1280` (output of `02_yolo`) with Faster R-CNN, no GPU needed.

In [ ]:
if "rewbf" in STAGES:
    y_val = C.load_preds(P.find_input("**/val_preds_yolov8s_1280.pkl"))
    y_test = C.load_preds(P.find_input("**/test_preds_yolov8s_1280.pkl"))
    fr_val = C.load_preds(P.find_input("**/val_preds_frcnn_base.pkl"))
    fr_test = C.load_preds(P.find_input("**/test_preds_frcnn_base.pkl"))
    for w in [(2, 1), (3, 1)]:
        log("rewbf", f"WBF yolov8s_1280@1280:frcnn={w[0]}:{w[1]}", P.wbf_ensemble([y_val, fr_val], list(w), sizes),
            weights=f"{w[0]}:{w[1]}")
    e = C.results_table(LOG).query("stage == 'rewbf'").set_index("weights")["mAP"]
    w = [int(x) for x in e.idxmax().split(":")]
    submit("wbf_yolo1280_frcnn", P.wbf_ensemble([y_val, fr_val], w, sizes), P.wbf_ensemble([y_test, fr_test], w, sizes))

## Results of this session

In [ ]:
res = C.results_table(LOG)
display(res)
fig, ax = plt.subplots(figsize=(8, 0.45 * len(res) + 1))
ax.barh(res["exp"], res["mAP"]); ax.invert_yaxis(); ax.set_xlabel("val mAP")
for y, v in enumerate(res["mAP"]):
    ax.text(v, y, f" {v:.4f}", va="center", fontsize=8)
plt.tight_layout(); plt.savefig(f"{WORK}/pipeline_results.png", dpi=120); plt.show()